# Walmart Store Sales — XGBoost + SARIMA baseline

Leakage-safe baseline following the repository XGBoost and SARIMA workflow.

In [ ]:
%pip install -q "xgboost>=3,<4" "statsmodels>=0.14,<1" "scikit-learn>=1.6,<2" "wandb>=0.19,<1" "cloudpickle>=3,<4"

In [ ]:
from __future__ import annotations

import json
import math
import os
import platform
import random
import time
import warnings
from pathlib import Path

import cloudpickle
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import wandb
import xgboost as xgb
from sklearn.base import BaseEstimator, RegressorMixin, TransformerMixin
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from statsmodels.tsa.statespace.sarimax import SARIMAX

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)

print({"python": platform.python_version(), "pandas": pd.__version__, "scikit_learn": sklearn.__version__, "xgboost": xgb.__version__, "wandb": wandb.__version__})


## Configuration

In [ ]:
DATA_DIR = Path("/content/drive/MyDrive/walmart_competition_data")
OUTPUT_DIR = Path("/content/artifacts/xgboost_sarima")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WANDB_ENTITY = "kende23-n-a"
WANDB_PROJECT = "Walmart-Recruiting---Store-Sales-Forecasting"
CONFIG = {
    "seed": SEED, "validation_weeks": 39, "holiday_weight": 5.0,
    "sarima_order": (1, 0, 1), "seasonal_order": (0, 1, 1, 52),
    "min_sarima_points": 80, "device": "cuda",
}
XGB_PARAMS = {"n_estimators": 1200, "learning_rate": 0.03, "max_depth": 9, "min_child_weight": 8, "subsample": 0.85, "colsample_bytree": 0.80, "reg_lambda": 4.0, "objective": "reg:absoluteerror", "tree_method": "hist", "device": CONFIG["device"], "random_state": SEED, "n_jobs": -1}


## Load and validate competition data

In [ ]:
TYPE_MAP = {"A": 0, "B": 1, "C": 2}
FEATURE_COLUMNS = ["Store", "Dept", "IsHoliday", "Type", "Size", "Year", "Month", "WeekOfYear", "WeekSin", "WeekCos"]


def merge_tables(sales, external, stores):
    return (sales.merge(external.drop(columns="IsHoliday"), on=["Store", "Date"], how="left", validate="many_to_one")
            .merge(stores, on="Store", how="left", validate="many_to_one"))


def build_features(frame):
    result = frame.copy()
    result["Type"] = result["Type"].map(TYPE_MAP).fillna(-1)
    week = result["Date"].dt.isocalendar().week.astype(int)
    result["Year"] = result["Date"].dt.year
    result["Month"] = result["Date"].dt.month
    result["WeekOfYear"] = week
    result["WeekSin"] = np.sin(2 * np.pi * week / 52.0)
    result["WeekCos"] = np.cos(2 * np.pi * week / 52.0)
    return result[FEATURE_COLUMNS].astype(float)


def weighted_mae(y_true, y_pred, is_holiday, holiday_weight=5.0):
    weights = np.where(np.asarray(is_holiday, dtype=bool), holiday_weight, 1.0)
    return float(np.average(np.abs(np.asarray(y_true) - np.asarray(y_pred)), weights=weights))


def fit_series_sarima(history, target, order, seasonal_order, min_points=80):
    predictions = np.full(len(target), np.nan)
    for key, group in target.groupby(["Store", "Dept"], sort=False):
        series = history[(history.Store == key[0]) & (history.Dept == key[1])].sort_values("Date")["Weekly_Sales"]
        if len(series) < min_points:
            continue
        try:
            fitted = SARIMAX(series, order=order, seasonal_order=seasonal_order, enforce_stationarity=False, enforce_invertibility=False).fit(disp=False, maxiter=60)
            predictions[target.index.get_indexer(group.index)] = fitted.forecast(len(group))
        except Exception:
            continue
    return predictions


In [ ]:
required_files = ["train.csv", "test.csv", "features.csv", "stores.csv"]
missing = [name for name in required_files if not (DATA_DIR / name).exists()]
assert not missing, f"Missing files: {missing}"
train_raw = pd.read_csv(DATA_DIR / "train.csv", parse_dates=["Date"])
test_raw = pd.read_csv(DATA_DIR / "test.csv", parse_dates=["Date"])
features_raw = pd.read_csv(DATA_DIR / "features.csv", parse_dates=["Date"])
stores_raw = pd.read_csv(DATA_DIR / "stores.csv")
train = merge_tables(train_raw, features_raw, stores_raw)
test = merge_tables(test_raw, features_raw, stores_raw)
assert not train.duplicated(["Store", "Dept", "Date"]).any()
print({"train_rows": len(train), "test_rows": len(test), "train_start": train.Date.min(), "train_end": train.Date.max()})


## Chronological validation split

In [ ]:
dates = np.sort(train["Date"].unique())
validation_start = pd.Timestamp(dates[-CONFIG["validation_weeks"]])
train_part = train[train.Date < validation_start].reset_index(drop=True)
valid_part = train[train.Date >= validation_start].reset_index(drop=True)
X_train = build_features(train_part)
X_valid = build_features(valid_part)
y_train = train_part["Weekly_Sales"]
y_valid = valid_part["Weekly_Sales"]
train_weights = np.where(train_part.IsHoliday, CONFIG["holiday_weight"], 1.0)
assert train_part.Date.max() < valid_part.Date.min()
print({"train_rows": len(train_part), "valid_rows": len(valid_part), "validation_start": validation_start})


## Train XGBoost and SARIMA

In [ ]:
run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, name="xgboost-sarima-baseline", job_type="train", tags=["baseline", "xgboost", "sarima", "wmae"], config={**CONFIG, **XGB_PARAMS})
model = xgb.XGBRegressor(**XGB_PARAMS)
model.fit(X_train, y_train, sample_weight=train_weights, eval_set=[(X_valid, y_valid)], verbose=100)
xgb_pred = model.predict(X_valid)
sarima_pred = fit_series_sarima(train_part, valid_part, CONFIG["sarima_order"], CONFIG["seasonal_order"], CONFIG["min_sarima_points"])
sarima_pred = np.where(np.isfinite(sarima_pred), sarima_pred, xgb_pred)
trials = []
for xgb_weight in np.arange(0.50, 0.91, 0.05):
    prediction = xgb_weight * xgb_pred + (1 - xgb_weight) * sarima_pred
    trials.append({"xgb_weight": float(xgb_weight), "wmae": weighted_mae(y_valid, prediction, valid_part.IsHoliday)})
blend_results = pd.DataFrame(trials).sort_values("wmae")
best_weight = float(blend_results.iloc[0]["xgb_weight"])
best_pred = best_weight * xgb_pred + (1 - best_weight) * sarima_pred
metrics = {"validation/xgboost_wmae": weighted_mae(y_valid, xgb_pred, valid_part.IsHoliday), "validation/sarima_wmae": weighted_mae(y_valid, sarima_pred, valid_part.IsHoliday), "validation/hybrid_wmae": weighted_mae(y_valid, best_pred, valid_part.IsHoliday), "validation/best_xgb_weight": best_weight}
run.log(metrics); run.log({"validation/blend_search": wandb.Table(dataframe=blend_results)}); run.summary.update(metrics)
display(pd.Series(metrics, name="value").to_frame()); display(blend_results)


## Diagnostics

In [ ]:
validation_results = valid_part[["Store", "Dept", "Date", "IsHoliday", "Weekly_Sales"]].copy()
validation_results["Prediction"] = best_pred
validation_results["AbsoluteError"] = (validation_results.Weekly_Sales - best_pred).abs()
display(validation_results.groupby("IsHoliday")["AbsoluteError"].agg(["mean", "median", "count"]))
run.log({"validation/predictions": wandb.Table(dataframe=validation_results.head(10000))})
run.finish()
